# **Laboratorio 3**
### ILI-257 PROGRAMACIÓN PARALELA APLICADA
### Profesor: Jorge Díaz M.
### Fecha: 30 de Abril de 2026

Instrucciones: Para esta evaluación debe desarrollar los kernels solicitados en cada una de las preguntas que se presentan a continuación. En cada uno de ellos debe completar en la sección de comentarios explicando que fue lo que hizo junto con el por que se necesitaba dicha solución.

Funciones generales Python utilizadas en esta evaluación.
## ¡No modificar!

In [25]:
from PIL import Image

def png_a_txt(input_png, output_txt):
    img = Image.open(input_png).convert("RGB")  # aseguramos RGB
    width, height = img.size
    pixels = list(img.getdata())  # [(R,G,B), (R,G,B), ...]

    with open(output_txt, "w") as f:
        f.write(f"{width} {height}\n")
        for (r, g, b) in pixels:
            f.write(f"{r} {g} {b} ")

    print(f"Imagen convertida a {output_txt}")

def txt_a_png(input_txt, output_png):
    with open(input_txt, "r") as f:
        width, height = map(int, f.readline().split())
        data = list(map(int, f.read().split()))

    pixels = []
    for i in range(0, len(data), 3):
        pixels.append((data[i], data[i+1], data[i+2]))

    img = Image.new("RGB", (width, height))
    img.putdata(pixels)
    img.save(output_png)

    print(f"Imagen guardada en {output_png}")

## Ejercicio 1 (40pts)



En este primer ejercicio se le pide que, dada una imagen RGB de tamaño potencia de 2 en formato TXT debe contar la cantidad total de pixeles que tiene el color rojo, verde y azul sobre un determinado valor alpha, beta y gamma (un contador por separado para rojo, verde y azul).

Para esto, debe implementar una versión secuencial de esto en CPU (No usando la GPU), y luego dos alternativas para realizar esto en GPU. Posteriormente, debe medir los tiempos de ejecución de cada una de esas alternativas, anotarlos y comentar cual es la mejor alternativa y por que cree que lo es. Además, para las alternativas en GPU, debe detectar cual es el número de hebras por bloque que da el mejor desempeño. Probar tamaños de bloque 16, 32, 64, 128, 256, 512.


Recuerde convertir su imagen "input.png" en TXT ejecutando la siguiente linea

In [26]:
png_a_txt("input.png", "input.txt")

Imagen convertida a input.txt


In [27]:
%%writefile image_edit.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <cuda_runtime.h>

using namespace std;

bool leerTXT(const string& filename, unsigned char*& data, int& width, int& height) {
    ifstream file(filename);
    if (!file) {
        cerr << "Error abriendo archivo\n";
        return false;
    }
    file >> width >> height;
    int size = width * height * 3;
    data = new unsigned char[size];
    for (int i = 0; i < size; i++) {
        int val;
        file >> val;
        data[i] = (unsigned char)val;
    }
    return true;
}

bool escribirTXT(const string& filename, unsigned char* data, int width, int height) {
    ofstream file(filename);
    if (!file) {
        cerr << "Error escribiendo archivo\n";
        return false;
    }
    file << width << " " << height << "\n";
    int size = width * height * 3;
    for (int i = 0; i < size; i++) {
        file << (int)data[i] << " ";
    }
    return true;
}

// Implementación secuencial en CPU:
bool filtro_en_CPU(unsigned char* input, unsigned char* output, int width, int height, int r){
  int size = width * height * 3;
  int ventana = 2*r+1;
  int divisor = ventana * ventana;

  for (int i = 0; i < size; i++) {
    int color = i % 3; // Si == 0 rojo, == 1 verde, == 2 azul
    int pixel_actual = i / 3; //Dado que usamos el size incluyendo los pixeles separados por RGB
    int x = pixel_actual % width; // Coordenada x sin rgb
    int y = pixel_actual / width;
    int suma = 0;

    for (int j = -r; j <= r; j++) { // Iteración en y
      for (int k = -r; k <= r; k++) { // Iteración en x
        // Las siguientes coords son en base al pixel no separado en rgb
        int x_f = x + k; // Coordenadas x del vecino
        int y_f = y + j; // Coordenadas y del vecino

        if (x_f < 0 || x_f >= width || y_f < 0 || y_f >= height) {
          continue; // Los píxeles fuera de la imagen aportan 0.
        }

        int vecino = ((y_f * width + x_f) * 3) + color;
        suma = suma + input[vecino];
      }
    }

    int pixel = suma / divisor;
    if (pixel > 255) {
      pixel = 255;
    }
    output[i] = pixel;
  }
  return true;
}

// __global__ void kernel_GPU1(unsigned char* input, unsigned char* output, int size) {
//     int idx = blockIdx.x * blockDim.x + threadIdx.x;

//     if (idx < size) {
//     }
// }

int main() {
    cudaError_t err = cudaSuccess;
    string inputFile = "input.txt";
    string outputFile = "output.txt";
    string outputFileCPU = "output_cpu.txt";

    int width, height;
    unsigned char* h_input = nullptr;
    unsigned char* h_output = nullptr;
    unsigned char* h_output_cpu = nullptr;

    if (!leerTXT(inputFile, h_input, width, height)) {
        return -1;
    }
    int numPixels = width * height;
    int size = numPixels* 3;
    h_output = new unsigned char[size];
    h_output_cpu = new unsigned char[size];

    unsigned char *d_input, *d_output;
    err = cudaMalloc((void **)&d_input, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_input (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMalloc((void **)&d_output, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_output (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMemcpy(d_input, h_input, size * sizeof(unsigned char), cudaMemcpyHostToDevice);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy d_input from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    int threads = 256;
    int blocks = (numPixels + threads - 1) / threads;


    int r = 1; // Tamaño de la ventana, se puede modificar para r = 1, 2 ,3.

    filtro_en_CPU(h_input, h_output_cpu, width, height, r);
    escribirTXT(outputFileCPU, h_output_cpu, width, height);

    //kernel<<<blocks, threads>>>(d_input, d_output, numPixels);
    cudaDeviceSynchronize();
    err = cudaMemcpy(h_output, d_output, size * sizeof(unsigned char) , cudaMemcpyDeviceToHost);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy h_output from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    escribirTXT(outputFile, h_output, width, height);

    cudaFree(d_input);
    cudaFree(d_output);
    delete[] h_input;
    delete[] h_output;
    delete[] h_output_cpu;

    cout << "Procesamiento RGB listo -> " << outputFile << " y " << outputFileCPU << endl;
    return 0;

}

Overwriting image_edit.cu


Recuerde convertir su imagen resultante en PNG ejecutando la siguiente linea

In [28]:
!nvcc -arch=sm_75 image_edit.cu -o image_edit
!./image_edit
print("\n")
txt_a_png("output1.txt", "output1.png")
txt_a_png("output1_gpu2.txt", "output1_gpu2.png")

Colores contabilizados en CPU:
 Rojo = 503512, Verde = 762765, Azul = 736304
Tiempo secuencial en CPU: 11.2729 ms

Colores contabilizados en GPU, forma N°1:
 Rojo = 503512, Verde = 762765, Azul = 736304
Tiempo en GPU forma 1: 0.072689 ms

Colores contabilizados en GPU, forma N°2:
 Rojo = 503512, Verde = 762765, Azul = 736304
Tiempo en GPU forma 2: 0.140702 ms

Procesamiento RGB listo -> output1.txt y output1_gpu2.txt


Imagen guardada en output1.png
Imagen guardada en output1_gpu2.png


## Respuesta
#### Debe explicar lo que hizo y por que lo hizo. Puede extenderse tanto como necesite

- Para hacerlo vía CPU secuencial simplemente se analizó el .txt con los colores de la imagen y se comparó cada uno de ellos (dependiendo su color, el cual se obtiene mediante la operación módulo) si están debajo del umbral definido para cada uno de ellos, para el rojo es alpha, para el verde es beta y para el azul es gamma. Dichos parámetros son totalmente modificables y se aplicarán a cada ejecución de este ejercicio N°1.  
Para la primera versión de la contabilización de los colores vía GPU se optó por hacerlo siguiendo las formas de los previos laboratorios, en el cual primero por cada hebra se verifica un pixel de la imagen, en él se separa el pixel por colores y se verifican si están o no abajo del umbral definido por alpha, beta y gamma. Si están debajo se marcan en el txt de output como 255, con el objetivo de poder contabilizar solo los que cumplieron las restricciones, además de poder observar la imagen resultante por dichas restricciones.  
Para la segunda versión se decidió utilizar una hebra por cada color de la imagen. Funciona de forma similar a la anterior, solo que esta vez se utilizarían muchas más hebras por cada imagen (y pixel podríamos decir). Se hizo así para comprobar la capacidad de utilización de las hebras por parte de Google Colab y medir su rendimiento tal y como pide el ejercicio.

## Ejercicio 2 (60pts)

En este segundo ejercicio se le pide implementar un filtro de procesamiento sobre una imagen RGB de tamaño potencia de 2, entregada en formato TXT.

El filtro a aplicar corresponde a una **convolución 2D (stencil)** sobre la imagen, utilizando una máscara cuadrada de tamaño $(2r + 1) \times (2r + 1)$, donde $r$ es un parámetro entregado al kernel CUDA.

Para cada pixel de salida, el valor de cada canal se calcula de forma independiente como:

$$
O_C(x,y) = \sum_{i=-r}^{r} \sum_{j=-r}^{r} I_C(x+i, y+j) \cdot K(i,j)
$$

donde:
- $C \in \{R, G, B\}$
- $I_C$ representa el canal correspondiente de la imagen de entrada
- $K$ es el kernel de convolución entregado como entrada

Los $K$ que se utilizarán serán dos:
- Promedio simple o suavizado 
$$K =
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$
- Edge Detection (o ejemplo lLaplaciano):  
$$K =
\begin{bmatrix}
-1 & -1 & -1 \\
-1 &  8 & -1 \\
-1 & -1 & -1
\end{bmatrix}
$$

Para esto, debe implementar una versión secuencial de esto en CPU (sin ejecución en GPU) y luego implementar las siguientes dos versiones en GPU:

- Versión GPU A:  Cada hebra calcula un pixel de salida (R, G y B) recorriendo toda la ventana de $(2r + 1)² $
- Versión GPU B: La misma definición de convolución que la versión GPU A, pero con una implementación optimizada. Se espera que reduzca el número de accesos a memoria global y evite cálculos redundantes dentro de los bucles internos (como el cálculo repetido de índices).


Ambas versiones deben mantener el mismo modelo de paralelización (1 hebra = 1 pixel de salida), variando únicamente la eficiencia de la implementación en el uso de memoria y cómputo dentro del kernel.


Para ambas versiones GPU probar tamaños de bloque 16, 32, 64, 128, 256, 512. Además, varie el R entre 1 a 3.

Recuerde medir los tiempos de las tres alternativas junto con determinar cuál entrega mejor rendimiento. Explique cómo cree que influye el tamaño del bloque en el rendimiento.

Nota: Para los pixeles fuera del rango de la imagen, asuma que sus valores RGB son 0.


Recuerde convertir su imagen "input.png" en TXT ejecutando la siguiente linea

In [29]:
png_a_txt("input.png", "input.txt")

Imagen convertida a input.txt


In [ ]:
%%writefile image_edit.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <cuda_runtime.h>

using namespace std;

bool leerTXT(const string& filename, unsigned char*& data, int& width, int& height) {
    ifstream file(filename);
    if (!file) {
        cerr << "Error abriendo archivo\n";
        return false;
    }
    file >> width >> height;
    int size = width * height * 3;
    data = new unsigned char[size];
    for (int i = 0; i < size; i++) {
        int val;
        file >> val;
        data[i] = (unsigned char)val;
    }
    return true;
}

bool escribirTXT(const string& filename, unsigned char* data, int width, int height) {
    ofstream file(filename);
    if (!file) {
        cerr << "Error escribiendo archivo\n";
        return false;
    }
    file << width << " " << height << "\n";
    int size = width * height * 3;
    for (int i = 0; i < size; i++) {
        file << (int)data[i] << " ";
    }
    return true;
}

// Implementación secuencial en CPU:
bool filtro_en_CPU(unsigned char* input, unsigned char* output, int width, int height, int r){
  int size = width * height * 3;
  int ventana = 2*r+1;

  for (int i = 0; i < 2; i++) {    
    int x = i % (width*3);
    int y = i / (width*3);
    int x_f = 0;
    int y_f = 0;
    int pixel = 0;
    printf("Pixel %d, coordenadas: (%d, %d)\n", i, x, y);
    color = i % 3;
    for (int j = -r; j <= r; j++) { // Iteración en y
      for (int k = -r; k <= r; k++) { // Iteración en x
        x_f = x + k*3;
        y_f = y + j;
        if (x_f < 0 || x_f >= width*3 || y_f < 0 || y_f >= height) {
          continue; // Ignorar píxeles fuera de los límites
        }
        pixel = pixel + input[y_f * width + x_f];
        printf("Pixel vecino: (%d, %d)\n", x_f, y_f);
      }
    }
    pixel = pixel * (1/(ventana * ventana));
    if (pixel > 255) {
      pixel = 255;
    }
    output[i] = pixel;
  }
//   printf("Colores contabilizados en CPU:\n Rojo = %d, Verde = %d, Azul = %d\n", r, g, b);
  return true;
}

// __global__ void kernel(unsigned char* input, unsigned char* output, int size) {
//     int idx = blockIdx.x * blockDim.x + threadIdx.x;

//     if (idx < size) {
//     }
// }

int main() {
    cudaError_t err = cudaSuccess;
    string inputFile = "input.txt";
    string outputFile = "output.txt";
    string outputFileCPU = "output_cpu.txt";

    int width, height;
    unsigned char* h_input = nullptr;
    unsigned char* h_output = nullptr;
    unsigned char* h_output_cpu = nullptr;

    if (!leerTXT(inputFile, h_input, width, height)) {
        return -1;
    }
    int numPixels = width * height;
    int size = numPixels* 3;
    h_output = new unsigned char[size];
    h_output_cpu = new unsigned char[size];

    unsigned char *d_input, *d_output;
    err = cudaMalloc((void **)&d_input, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_input (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMalloc((void **)&d_output, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_output (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMemcpy(d_input, h_input, size * sizeof(unsigned char), cudaMemcpyHostToDevice);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy d_input from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    int threads = 256;
    int blocks = (numPixels + threads - 1) / threads;


    int r = 1; // Tamaño de la ventana, se puede modificar para r = 1, 2 ,3.

    filtro_en_CPU(h_input, h_output_cpu, width, height, r);
    escribirTXT(outputFileCPU, h_output_cpu, width, height);

    //kernel<<<blocks, threads>>>(d_input, d_output, numPixels);
    cudaDeviceSynchronize();
    err = cudaMemcpy(h_output, d_output, size * sizeof(unsigned char) , cudaMemcpyDeviceToHost);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy h_output from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    escribirTXT(outputFile, h_output, width, height);

    cudaFree(d_input);
    cudaFree(d_output);
    delete[] h_input;
    delete[] h_output;
    delete[] h_output_cpu;

    cout << "Procesamiento RGB listo -> " << outputFile << " y " << outputFileCPU << endl;
    return 0;

}

Overwriting image_edit.cu


Recuerde convertir su imagen resultante en PNG ejecutando la siguiente linea

In [41]:
!nvcc -arch=sm_75 image_edit.cu -o image_edit
!./image_edit
txt_a_png("output.txt", "output2.png")
txt_a_png("output_cpu.txt", "output_cpu.png")


image_edit.cu(51): error: identifier "color" is undefined
      color = i % 3;
      ^

image_edit.cu(95): error: identifier "h_output_cpu" is undefined
      h_output_cpu = new unsigned char[size];
      ^

2 errors detected in the compilation of "image_edit.cu".
Pixel 0, coordenadas: (0, 0)
Pixel vecino: (0, 0)
Pixel vecino: (3, 0)
Pixel vecino: (0, 1)
Pixel vecino: (3, 1)
Pixel 1, coordenadas: (1, 0)
Procesamiento RGB listo -> output.txt
Imagen guardada en output2.png


FileNotFoundError: [Errno 2] No such file or directory: 'output_cpu.txt'

## Respuesta
#### Debe explicar lo que hizo y por que lo hizo. Puede extenderse tanto como necesite

-